GPU được kiểm tra sau khi cài dependencies pinned, bằng tiến trình Python mới ở cell smoke.
Chạy notebook từ đầu để mount Drive và cập nhật repo trước khi chạy experiment.


# 04D — Portable local / Colab setup and experiments

Default: **non-training smoke test**. Locally, use the current checkout and a pinned Python environment; set `EDGEAI_DATA_ROOT` before importing tools to use an external EdgeAI storage tree. Colab mounts Drive, clones or fast-forward pulls the GitHub repository, and installs its pinned runtime requirements. Code executes in the connected kernel's filesystem.

Upload the complete staged `EdgeAI` tree to `/content/drive/MyDrive/EdgeAI` and commit/push local code fixes before the first Colab run. Never run 00–03B to repair a missing cache. This notebook does not push code or upload data.


In [11]:
from pathlib import Path, PureWindowsPath
import os, subprocess, sys
try:
    from google.colab import drive
except ImportError:
    drive = None
IN_COLAB = drive is not None

REPO_URL = "https://github.com/ancaranovik/Mosquito_Wingbeat_v1.git"
if IN_COLAB:
    drive.mount("/content/drive")
    assert REPO_URL.startswith("https://github.com/") and "@" not in REPO_URL
    repo_hint = os.environ.get("EDGEAI_REPO_ROOT", "/content/edge-ai")
    if PureWindowsPath(repo_hint).drive:
        raise ValueError("Colab needs a runtime checkout, not a Windows repository path")
    repo = Path(repo_hint).resolve()
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    else:
        origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=repo, text=True).strip()
        assert origin == REPO_URL, "Existing checkout has a different origin"
        assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip(), "Preserve uncommitted runtime changes before pulling"
        subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
else:
    hint = Path(os.environ.get("EDGEAI_REPO_ROOT", Path.cwd())).resolve()
    repo = next((p for p in [hint, *hint.parents] if (p / "tools/project_paths.py").is_file()), None)
    if repo is None:
        raise RuntimeError("Set EDGEAI_REPO_ROOT to your local checkout")
os.environ["EDGEAI_REPO_ROOT"] = str(repo)
os.chdir(repo)
sys.path.insert(0, str(repo / "tools"))
sys.dont_write_bytecode = True
print("Mode:", "COLAB" if IN_COLAB else "LOCAL")
print("Checkout:", repo)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mode: COLAB
Checkout: /content/edge-ai


In Colab, install the pinned cache-consumer and notebook dependencies. Locally, use the already configured environment without installing packages. Verification and smoke testing use fresh child Python processes. Configure private GitHub authentication interactively; never embed credentials here.


In [12]:
from colab_bootstrap import configure
paths = configure(install=IN_COLAB)


Repository: /content/edge-ai
Data: /content/drive/MyDrive/EdgeAI
Immutable baseline: /content/drive/MyDrive/EdgeAI/fixed/baseline/stage04
Future results: /content/drive/MyDrive/EdgeAI/experiments/<experiment_id>


Verify all frozen identities and eight saved baseline runs. This reads saved predictions and hashes; it does not evaluate a trained model on TEST again or regenerate any data.

In [ ]:
subprocess.run([sys.executable, "-B", "tools/verify_portability.py"], cwd=repo, check=True)


Colab requires CUDA and reports the GPU name before eight synthetic forward checks. Local mode uses CPU. Neither mode constructs an optimizer or starts training.


In [13]:
device = "cuda" if IN_COLAB else "cpu"
subprocess.run([sys.executable, "-B", "tools/experiment_runner.py", "smoke", "--device", device], cwd=repo, check=True)
print("Future outputs:", paths.EXPERIMENT_ROOT / "<experiment_id>")


Future outputs: /content/drive/MyDrive/EdgeAI/experiments/<experiment_id>


## Lần tiếp theo: exp_007_power075 — 8 cấu hình, class-weight exponent 0.75

Chỉ đổi weighted CE thành `(N_train/(4*n_train_class))**0.75`.
Đối chứng chính **exp_004_train_norm** (exponent 1); 04C cũng đối chiếu
**exp_006_sqrt_weights** (exponent 0.5). Protocol v4 dùng đúng counts TRAIN windows
đã freeze; engine baseline/v2/v3 và cache evidence giữ nguyên.

Giữ Adam LR=0.001, train_global_zscore TRAIN-only, seed 42, batch 64,
tối đa 50 epoch, patience 8, kiến trúc, split/cache; không augmentation.
Chọn checkpoint theo validation Macro-F1; early stopping theo weighted validation
loss. **TEST tắt**. Weights đổi có thể làm đổi lịch dừng; không so trực tiếp loss
với exp004/exp006. Đánh giá F1 tổng, Squ precision/recall/F1, Ara/HBN và Culex/LT.

Tên và mô tả nhập tại cell dưới; exponent nằm trong JSON config.
Preview phải in exponent **0.75**, policy **power_inverse_frequency**, đủ 8 run.
Commit với RUN_TRAINING=False; sau setup + GPU smoke PASS và preview đúng,
đổi True trong bản điều khiển. Checkout Colab phải sạch: nếu notebook nằm trong
repo, dùng bản sao ngoài /content/edge-ai trước khi chỉnh/lưu output.

Một lần chạy = 4 model 03A rồi 4 model 03B; output mới:
`MyDrive/EdgeAI/experiments/exp_007_power075/`.
Tên đã tồn tại bị từ chối; retry dùng tên mới như exp_007_power075_retry01.
Không cần chạy lại 00–03B hoặc train riêng 04A/04B.
Sau COMPLETE 8/8, mở 04C cùng tên. Kết quả exp006 đã sao lưu nguyên bản tại
`_local_only/notebook_snapshots/before_exp007_20261002_005648/`.
Hướng dẫn: `docs/EXP_007_POWER075.md`.


In [ ]:
# CHỈNH ĐIỀU KHIỂN TẠI ĐÂY; mỗi lần train dùng tên chưa tồn tại.
EXPERIMENT_NAME = "exp_007_power075"
EXPERIMENT_DESCRIPTION = "Lần 7: class-weight exponent 0.75, đối chiếu exp_004_train_norm (1.0) và exp_006_sqrt_weights (0.5); giữ Adam LR=0.001, train_global_zscore TRAIN-only và các thông số khác. Cache giữ nguyên; TEST tắt."
EXPERIMENT_CONFIG = "configs/experiments/power075_v4.json"
RUN_TRAINING = False

from cache_consumer import read_json
import experiment_runner
from experiment_runner import run_plan, training_module
config_preview = read_json(repo / EXPERIMENT_CONFIG)
config_preview.update(experiment_id=EXPERIMENT_NAME, description=EXPERIMENT_DESCRIPTION)
protocol = config_preview.get("protocol_version")
if protocol and protocol not in getattr(experiment_runner, "VERSIONED_PROTOCOLS", ()):
    raise RuntimeError(
        f"Loaded runner does not support {protocol}. Pull the latest repo, restart the kernel, "
        "then rerun setup and preview with RUN_TRAINING=False. "
        f"Loaded module: {experiment_runner.__file__}"
    )
plan = run_plan(config_preview)
engine = training_module(config_preview)
effective = (engine.effective_config(config_preview["training_overrides"], config_preview["evaluate_test"])
             if "protocol_version" in config_preview else engine.CONFIG)
experiment_path = paths.experiment(EXPERIMENT_NAME)
print("Experiment:", EXPERIMENT_NAME)
print("Kết quả:", experiment_path)
print("Reference:", config_preview.get("reference_experiment_id", "accepted baseline"))
print("Additional comparison in 04C:", "exp_006_sqrt_weights")
print("Params thực thi:", effective)
if hasattr(engine, "class_weight_record"):
    from cache_consumer import CLASSES, COUNTS, TRAIN_COUNTS
    accepted_weights = {label: COUNTS["train"] / (4 * count)
                        for label, count in zip(CLASSES, TRAIN_COUNTS)}
    print("TRAIN class weights:", engine.class_weight_record(effective, accepted_weights))
print("Kế hoạch:", len(plan), "cấu hình, chạy lần lượt:", [p["run_id"] for p in plan])
if RUN_TRAINING:
    subprocess.run([
        sys.executable, "-u", "-B", "tools/experiment_runner.py", "train",
        "--config", EXPERIMENT_CONFIG, "--experiment-id", EXPERIMENT_NAME,
        "--description", EXPERIMENT_DESCRIPTION,
    ], cwd=repo, check=True)
else:
    print("Training disabled. Kiểm tra tên/config/params, rồi đổi RUN_TRAINING thành True khi muốn chạy.")


## Xem kết quả sau khi đủ 8 cấu hình hoàn tất

Cell này chỉ đọc/xác thực kết quả; không train hoặc chạy lại TEST. Trong 04C, đặt cùng `EXPERIMENT_NAME` để xem history và confusion matrix của cả 8 cấu hình.


In [ ]:
if RUN_TRAINING:
    from experiment_review import compare_saved_results
    from IPython.display import display
    display(compare_saved_results(EXPERIMENT_NAME))
else:
    print("Để xem lần đã chạy, mở 04C và nhập EXPERIMENT_NAME tương ứng; không cần bật training.")
